# Pump and Treat - Well System Model

### Problem definition 

Strategy 2: Control the well rates.

## pmyf6 dynamic control 

The wells will be pumping 24h and regulated by two key thresholds: environmental quality standards for water of 0.1 mg/L (He et al., 2024) and maximum rate per well was 50 m3/d (Song et al., 2024). To increase efficiency, we vary the rate of the pumping wells since optimization is crucial to balance treatment speed, cost and water use.
The technical objective is to dynamically regulate the wells based on the values at the observation well, located at the area that we want to protect at the municipal boundary. The state has regulated thresholds for the amount that is possible to extract and treat. That amount can be reached or not. By using a control script, we optimize by regulating the minimum rate needed to maintain the water quality bellow the threshold and implies that we will treat the optimum amount. 

### Magic commands - auto reload of the model each time 

In [55]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
%autoreload 2

### Import from pymf6tools the functions to run, get and visualize simulation results

In [57]:
from pathlib import Path 
from pymf6.mf6 import MF6
import pandas as pd 
from functools import partial 
import numpy as np 

from pymf6_tools.make_model import run_simulation, get_simulation


from pymf6_tools.base_model import make_model_data
from pymf6_tools.make_model import make_input, run_simulation, get_simulation
from pymf6_tools.plotting import show_heads, show_well_head, show_bcs

In [58]:
from pymf6_tools.plotting import (
show_heads, show_well_head, show_concentration, show_bcs, 
show_bot_elevations, show_river_stages, contour_bot_elevations, 
plot_spec_discharge)
import pymf6_tools

## Set model path and name 

In [61]:
model_path = r'models/pymf6/pumptreat'
model_name = 'pumptreat'

### Run model 

In [63]:
run_simulation(model_path, verbosity_level=1)

loading simulation...
  loading simulation name file...
  loading tdis package...
  loading model gwf6...
    loading package dis...
    loading package ic...
    loading package npf...
    loading package sto...
    loading package chd...
    loading package wel...
    loading package oc...
  loading model gwt6...
    loading package dis...
    loading package ic...
    loading package adv...
    loading package dsp...
    loading package mst...
    loading package ssm...
    loading package cnc...
    loading package oc...
  loading exchange package gwf-gwt_exg_0...
  loading solution package gwf_pumptreat...
  loading solution package gwt_pumptreat...
FloPy is using the following executable to run the model: ..\..\..\..\..\..\..\mf6.6.2_win64\bin\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.6.2 05/12/2025

   MODFLOW 6 compiled May 12 2025 12:42:18 with Intel(R) Fortran Inte

In [34]:
from pymf6.mf6 import MF6

In [35]:
mf6 = MF6(model_path)

In [36]:
mf6.models.keys()

dict_keys(['gwf6', 'gwt6'])

In [37]:
gwf_models = mf6.models['gwf6']

In [38]:
gwf_models.keys()

dict_keys(['gwf_pumptreat'])

In [39]:
gwt_models = mf6.models['gwt6']

In [40]:
gwt_models.keys()

dict_keys(['gwt_pumptreat'])

### Inspect model packages 

In [41]:
gwt = gwt_models['gwt_pumptreat']

In [42]:
gwf = gwf_models['gwf_pumptreat']

In [43]:
gwf.packages

,description,is_mutable
name,,
dis,DIS Package: DIS,False
mvr,MVR Package: MVR,True
vsc,VSC Package: VSC,True
buy,BUY Package: BUY,True
ic,IC Package: IC,False
gnc,GNC Package: GNC,True
sto,STO Package: STO,False
npf,NPF Package: NPF,False
csub,CSUB Package: CSUB,True


In [44]:
gwt.packages

,description,is_mutable
name,,
dis,DIS Package: DIS,False
mst,MST Package: MST,False
mvt,MVT Package: MVT,True
adv,ADV Package: ADV,True
dsp,DSP Package: DSP,False
ssm,SSM Package: SSM,True
cnc,CNC Package: CNC,True
ic,IC Package: IC,False
fmi,FMI Package: FMI,True


### Inspect well package 

In [45]:
for _ in mf6.model_loop():
    if gwf.kper > 0:
        break

In [46]:
wel = gwf.packages.get_package('wel-1').as_mutable_bc()

In [47]:
wel.nodelist[:]

0    (0, 40, 25)
1    (0, 33, 25)
2    (0, 25, 25)
Name: nodelist, dtype: object

### Inspect values at any node

In [48]:
initial_head = gwf.X[(0, 41, 26)]
initial_head

np.float64(23.097009829214954)

In [49]:
conc = gwt.X[(0, 41, 26)]
conc

np.float64(152.44014664187887)

### Forward it to next time-step 2

In [50]:
for _ in mf6.model_loop():
    if gwf.kper == 1 and gwf.kstp > 245:
        break

### Concentration at sources

In [51]:
gwt.X

array([[[6.25970264e-13, 1.10727126e-12, 1.81836197e-12, ...,
         2.24684533e-23, 1.21597102e-23, 6.39720051e-24],
        [9.15920648e-13, 1.61606395e-12, 2.64381374e-12, ...,
         2.70411122e-23, 1.46014694e-23, 7.66944854e-24],
        [1.63768661e-12, 2.88009255e-12, 4.68922443e-12, ...,
         3.69725594e-23, 1.98857938e-23, 1.04196401e-23],
        ...,
        [1.86292625e-17, 3.34171791e-17, 5.61192575e-17, ...,
         2.30620191e-27, 1.09477616e-27, 5.28750957e-28],
        [1.09715923e-17, 1.96954997e-17, 3.31193268e-17, ...,
         1.40888288e-27, 6.67291970e-28, 3.21783623e-28],
        [7.79681764e-18, 1.40055802e-17, 2.35776954e-17, ...,
         1.01789961e-27, 4.81313370e-28, 2.31829079e-28]]],
      shape=(1, 101, 101))

In [52]:
conc = gwt.X[(0, 41, 26)]
conc

np.float64(389.4107673239013)

### Concentration at observation well 

In [53]:
conc = gwt.X[(0, 56, 26)]
conc

np.float64(0.04156293207649238)

## Run the control script 

In [54]:
# %load system_well_control.py
from pymf6.mf6 import MF6
import os
import pandas as pd

from matplotlib import pyplot as plt

def run_model(model_path, verbose=False):
    """Control script to regulate the pumping wells dynamically. The regulation of the system is regulated by
     concentration threshold at the obesrvation well, the groundwater mimimum threshold and volume of water to treat. """
    print('started')

    # Initialize the MF6 model using the provided nam file
    mf6 = MF6(model_path)
    print('mf6 initialized')

    # Get the flow models
    flow_models = mf6.models['gwf6']
    gwf = flow_models['gwf_pumptreat'] # Flow model name
    transport_models = mf6.models['gwt6']
    gwt = transport_models['gwt_pumptreat'] # Transport model name

    # Get well package
    for _ in mf6.model_loop():
        if gwf.kper > 0: # break after
            break

    wel = gwf.packages.get_package('wel-1').as_mutable_bc()
    well_coords = wel.nodelist[:]
    well_node_obs_coords = wel.nodelist[0]
    well_regulated_1_coords =  wel.nodelist[1]
    well_regulated_2_coords =  wel.nodelist[2]
    initial_head = gwf.X[well_node_obs_coords]
    print (initial_head)
    
    # Concentration control parameters
    tolerance_conc = 0.05
    conc_limit = 0.1 
    lower_limit_conc = conc_limit - tolerance_conc
    upper_limit_conc = conc_limit + tolerance_conc

    # Set limits for pumping rate 
    min_rate = -0.05  # Minimum extraction rate (m³/day)
    max_rate = -50.0  # Maximum extraction rate (m³/day)

    # State tracking variables
    below_gw = False
    above_conc = False
    
   # print("Regulated wells:", well_regulated)
    mywell_q = {
        'step': [],
        'head': [],
        'conc': [],
        'source_conc':[],
        'q_well1': [],
        'q_well2': [],
        'q_well3': [],
        'head_state': [],
        'conc_state': [], 
        'vol_water':[]
    } # Dict of lists per well
    
    # Models parameters 
    N = 101 

    # Run the model loop
    for model in mf6.model_loop():
        if gwf.kper == 1:  # Only operate during stress period 2
            # concentration at the source 
            current_head = gwf.X[(0, 41, 26)]
            current_conc = gwt.X[(0, 41, 26)]
            # concentration at the observation well 
            current_head = gwf.X[(0, 56, 26)]
            obs_conc = gwt.X[(0, 56, 26)]
            daily_volume = - 12 * (wel.q[0] + wel.q[1] + wel.q[2])
            # print (current_conc)

            # Record system state
            mywell_q['step'].append(gwf.kstp)
            mywell_q['conc'].append(obs_conc)
            mywell_q['source_conc'].append(current_conc)
            mywell_q['head'].append(current_head)
            current_q = wel.q.copy()  # Get current rates
            print(current_q)
            mywell_q['q_well1'].append(wel.q[0])
            mywell_q['q_well2'].append(wel.q[1])
            mywell_q['q_well3'].append(wel.q[2])
            #mywell_q['t_vol'].append(gwf.kstp * wel.q[2]*)
            mywell_q['head_state'].append('below' if below_gw else 'normal')
            mywell_q['conc_state'].append('above' if above_conc else 'normal')
            mywell_q['vol_water'].append(daily_volume) # since lenght of each period is 12 days 
            print ('CONCENTRATION AT SOURCE IS', current_conc)
            print ('CONCENTRATION AT OBSERVATION WELL IS', obs_conc)
       # else: 

            # Concentration regulation
            if obs_conc >= upper_limit_conc:
                above_conc = True  
                print(wel.q)
                q = wel.q
                # Control for well 1
                q[0] = q[0] * 1.1
                q[1] = q[1] * 1.2
                q[2] = q[2] * 1.3
                # well control
                for i in range(3):
                    if q[i] < max_rate:
                        q[i] = max_rate
                    elif q[i] > min_rate:
                        q[i] = min_rate
                wel.q = q
                print(f"Step {gwf.kstp}: Conc above limit! Increase pumping")
                
            elif obs_conc <= lower_limit_conc:
                above_conc = False # reset state
                print(wel.q)
                q = wel.q
               # Control for well 1
                q[0] = q[0] * 0.9
                q[1] = q[1] * 0.7
                q[2] = q[2] * 0.8
                for i in range(3):
                    if q[i] < max_rate:
                        q[i] = max_rate
                    elif q[i] > min_rate:
                        q[i] = min_rate
                wel.q = q
                print(f"Step {gwf.kstp}: Conc recovered! Reduce pumping")

    # total volume 
    total_volume = sum(mywell_q['vol_water'])  # in m³
    print(f"Total volume extracted: {total_volume:.2f} m³")
    
    # Save results
    df = pd.DataFrame(mywell_q)
    df.to_csv("well_control_results.csv", index=False)
    print("Simulation completed")
    return mywell_q


def plot(mywell_q):
    """Simple visualization of results"""
    plt.figure(figsize=(10, 6))

    # Plot pumping rates
    plt.plot(mywell_q['step'], mywell_q['q_well1'], 'b-o', label='Well 1 Pumping')
    plt.plot(mywell_q['step'], mywell_q['q_well2'], 'g-o', label='Well 2 Pumping')
    plt.plot(mywell_q['step'], mywell_q['q_well3'], 'r-o', label='Well 3 Pumping')
    plt.ylabel("Pumping Rate")
    plt.xlabel("Timestep")
    plt.grid(True)
    plt.legend()

    # Plot head and concentration
    plt.twinx()
    plt.plot(mywell_q['step'], mywell_q['head'], 'g--', label='Head')
    plt.plot(mywell_q['step'], mywell_q['conc'], 'm--', label='Concentration')
    plt.ylabel("Head/Concentration")
    plt.legend()

    plt.title("Well Control Performance")
    plt.tight_layout()
    plt.savefig("well_control_plot.png", dpi=300)
    plt.show()


def plot_state(mywell_q):
    """Enhanced visualization with state tracking"""
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    
    # Plot 1: Pumping Rates
    ax1.plot(mywell_q['step'], mywell_q['q_well1'], 'b-', label='Well 1')
    ax1.plot(mywell_q['step'], mywell_q['q_well2'], 'r-', label='Well 2')
    ax1.plot(mywell_q['step'], mywell_q['q_well3'], 'g-', label='Well 3')
    ax1.set_ylabel("Pumping Rate [L³/T]")
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.set_title("Pumping Rates")
    
    # Plot 2: Head with state indicators
    ax2.plot(mywell_q['step'], mywell_q['head'], 'go', label='Head')
    #ax2.axhline(y=24.75, color='gray', linestyle='--', label='Target')
    #ax2.axhline(y=25, color='red', linestyle=':', alpha=0.5, label='Lower Limit')
    #ax2.axhline(y=24, color='blue', linestyle=':', alpha=0.5, label='Upper Limit')
    
    # Mark head below state
    for i, state in enumerate(mywell_q['head_state']):
        if state == 'below':
            ax2.axvline(x=mywell_q['step'][i], color='orange', alpha=0.2)
    
    ax2.set_ylabel("Head [m]")
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    ax2.set_title("Head at Observation Well")
    
    # Plot 3: Concentration with state indicators
    ax3.plot(mywell_q['step'], mywell_q['conc'], 'm-', label='Concentration')
    ax3.axhline(y=0.1, color='purple', linestyle='--', label='Target')
    ax3.axhline(y=0.05, color='pink', linestyle=':', alpha=0.7, label='Lower Limit')
    ax3.axhline(y=0.15, color='pink', linestyle=':', alpha=0.7, label='Upper Limit')
    
    # Mark conc above state
    for i, state in enumerate(mywell_q['conc_state']):
        if state == 'above':
            ax3.axvline(x=results['step'][i], color='red', alpha=0.2)
    
    ax3.set_xlabel("Timestep")
    ax3.set_ylabel("Concentration")
    ax3.grid(True, alpha=0.3)
    ax3.legend()
    ax3.set_title("Concentration at Observation Well")
    
    plt.suptitle("Well Control System Performance", fontsize=16)
    plt.tight_layout()
    plt.savefig("well_control_analysis.png", dpi=300)
    plt.show()

if __name__ == '__main__':
    model_path = os.path.join(os.getcwd(), 'models', 'pumptreat')
    results = run_model(model_path=model_path, verbose=False)
    plot(results)
    plot_state(results)

started


FileNotFoundError: C:\Users\lucialabarca\re-run noteboks\pymf6-validation\src\notebooks\models\pumptreat\mfsim.nam

### Why running the control script?

If there is no control over the pumping of the wells we would extract *450 150 m³*, with control the model we would extract *18 377.1167 m³*. 